In [1]:
#%run input/Format.ipynb
import ROOT as root
import math

root.gErrorIgnoreLevel = root.kWarning
root.gStyle.SetOptStat(0)
root.gStyle.SetOptFit(0)
%jsroot on


Welcome to JupyROOT 6.30/06


In [2]:
# ============================================================
# Input/output and all analysis choices
# ============================================================

safe_to_pdf = True

file_path = "input/"

# Add, remove, or reorder files here.
# The same histogram from every enabled file is drawn on the same pad.
input_samples = [
    {
        "name": "spacecharge_1st_order",
        "file_name": "v0s_pp_79513_newv0_seg_PR_full_pt02_r2_z15_q15_nc30_noCS0_v0.root",#v0s_pp_MYu_ratio_new_vtx_1p23_1p0_1p6_000_v14 v0s_pp_MYu_ratio_new_vtx_1p23_1p0_1p6_000_corrmapp_stream_full_v2 v0s_pp_MYu_ratio_new_vtx_1p23_1p0_1p6_000_stream_full_v0
        "label": "1st-order cor.",#"", cs1==cs2
        "color": root.kBlue + 1,
        "marker": 20,
        "enabled": True,
    },
    {
        "name": "correction_2nd_order",
        "file_name": "v0s_pp_79523_newv0_PR_full_pt02_r2_z15_q15_nc30_noCS0_v1.root",# v0s_pp_MYu_ratio_new_vtx_1p23_1p0_1p6_000_corrmapp_v7  v0s_pp_MYu_ratio_new_vtx_1p23_1p0_1p6_000_stream_full_corrmap_v0
        "label": "2nd-order cor.",#"no cs cut
        "color": root.kRed + 1,
        "marker": 24,
        "enabled": True,
    },#v0s_pp_MYu_ratio_new_vtx_1p23_1p0_1p6_000_corrmapp_v0 v0s_pp_MYu_ratio_new_vtx_1p23_1p0_1p6_000_corrmapp_v6 
    #v0s_pp_MYu_ratio_new_vtx_1p23_1p0_1p6_000_stream_full_v0 v0s_pp_MYu_ratio_new_vtx_1p23_1p0_1p6_000_stream_full_corrmap_v5

    # v0s_pp_MYu_ratio_new_vtx_1p23_1p0_1p6_000_stream_full_v0
    # v0s_pp_MYu_ratio_new_vtx_1p23_1p0_1p6_000_stream_full_corrmap_v0
    #v0s_pp_MYu_ratio_new_vtx_1p23_1p0_1p6_000_stream_full_v1
    #v0s_pp_MYu_ratio_new_vtx_1p23_1p0_1p6_000_stream_full_corrmap_v7
    #v0s_pp_79513_old_mfield_z15_q15_nc30_noCS0_v0 v0s_pp_79523_PR_full_pt02_r2_z15_q15_nc30_noCS0_v0
    #v0s_pp_79513_PR_full_pt02_r2_z15_q15_nc30_noCS0_v0 v0s_pp_79513_newv0_PR_full_pt02_r2_z15_q15_nc30_noCS0_v0 v0s_pp_79513_newv0_PR_full_pt02_r2_z15_q15_nc30_noCS0_v2
    #v0s_pp_79513_newv0_PR_full_pt02_r2_z15_q15_nc30_noCS0_corrmap_v0 pp_79523_PR_ifc_3D_Fm120_1p8_2p65_2p65_375V_kf_v1  v0s_pp_79516_newv0_PR_full_pt02_r2_z15_q15_nc30_noCS0_v0
    #v0s_pp_79513_newv0_seg_PR_full_pt02_r2_z15_q15_nc30_noCS0_v0
    #
    # Example of a third file:
    # {
    #     "name": "third_sample",
    #     "file_name": "third_file.root",
    #     "label": "third correction",
    #     "color": root.kGreen + 2,
    #     "marker": 21,
    #     "enabled": True,
    # },
]

# Drawing controls.
draw_scaled_like_sign = False
draw_total_fit = False
draw_signal_component = False
normalize_for_shape_comparison = False
draw_diff = False

outdir = "output/v0_mass_selected_comparison"
root.gSystem.mkdir(outdir, True)

data_set = "p+p 2025"
run_text = "Run 79513"

like_background_method = "combined_like"   # "combined_like" or "geometric"
ncols = 3

# Draw integrated pT first, then the differential pT bins.
integrated_pt = (0.5, 3.0)

# Selected histogram directories/cuts.
cut_k0s = "cut03_baseline"
cut_lambda = "cut03_baseline"
cut_antilambda = "cut03_baseline"
cut_phi = "promptMesonsPrimaryConstrained/primary_dz_0p5"
cut_d0 = "promptMesons/primary_dz_0p5_track"
cut_antid0 = "promptMesons/primary_dz_0p5_track"

pt_bins = {
    "k0s":       [0.5, 0.8, 1.1, 1.4, 1.8, 2.2, 3.0],
    "lambda":    [0.5, 0.8, 1.1, 1.4, 1.8, 2.2, 3.0],
    "antilambda":[0.5, 0.8, 1.1, 1.4, 1.8, 2.2, 3.0],
    "phi":       [1.1, 1.4, 1.8, 2.2, 3.0, 4.0, 5.0],
    "d0":        [1.1, 1.4, 1.8, 2.2, 3.0, 4.0, 5.0],
    "antid0":    [1.1, 1.4, 1.8, 2.2, 3.0, 4.0, 5.0],
}

species = {
    "k0s": {
        "label": "K^{0}_{S}",
        "mass_title": "m_{#pi^{+}#pi^{-}} [GeV/c^{2}]",
        "cut": cut_k0s,
        "hist": "h_mass_Kshort_vs_v0_pt",
        "draw_range": (0.40, 0.60),
        "sidebands": [(0.40, 0.44), (0.56, 0.60)],
        "seed_range": (0.475, 0.520),
        "fit_range": (0.43, 0.57),
        "mean_limits": (0.485, 0.510),
        "sigma_limits": (0.002, 0.030),
        "rebin": [1, 1, 1, 1, 1, 1],
        "model": "gaus",
    },
    "lambda": {
        "label": "#Lambda",
        "mass_title": "m_{p#pi^{-}} [GeV/c^{2}]",
        "cut": cut_lambda,
        "hist": "h_mass_Lambda_vs_v0_pt",
        "draw_range": (1.08, 1.15),
        "sidebands": [(1.080, 1.100), (1.135, 1.150)],
        "seed_range": (1.105, 1.127),
        "fit_range": (1.090, 1.145),
        "mean_limits": (1.108, 1.123),
        "sigma_limits": (0.001, 0.020),
        "rebin": [1, 1, 1, 1, 1, 1],
        "model": "gaus",
    },
    "antilambda": {
        "label": "#bar{#Lambda}",
        "mass_title": "m_{#bar{p}#pi^{+}} [GeV/c^{2}]",
        "cut": cut_antilambda,
        "hist": "h_mass_AntiLambda_vs_v0_pt",
        "draw_range": (1.08, 1.15),
        "sidebands": [(1.080, 1.100), (1.135, 1.150)],
        "seed_range": (1.105, 1.127),
        "fit_range": (1.090, 1.145),
        "mean_limits": (1.108, 1.123),
        "sigma_limits": (0.001, 0.020),
        "rebin": [1, 1, 1, 1, 1, 1],
        "model": "gaus",
    },
    "phi": {
        "label": "#phi",
        "mass_title": "m_{K^{+}K^{-}} [GeV/c^{2}]",
        "cut": cut_phi,
        "hist": "h_mass_Phi_vs_v0_pt",
        "draw_range": (0.98, 1.08),
        "sidebands": [(0.980, 1.000), (1.045, 1.080)],
        "seed_range": (1.005, 1.035),
        "fit_range": (0.990, 1.065),
        "mean_limits": (1.010, 1.030),
        "sigma_limits": (0.0005, 0.020),
        "rebin": [1, 1, 1, 2, 2, 3],
        "model": "voigt",
        "gamma": 0.004249,
    },
    "d0": {
        "label": "D^{0}",
        "mass_title": "m_{K^{-}#pi^{+}} [GeV/c^{2}]",
        "cut": cut_d0,
        "hist": "h_mass_D0_vs_v0_pt",
        "draw_range": (1.70, 2.05),
        "sidebands": [(1.70, 1.78), (1.95, 2.05)],
        "seed_range": (1.82, 1.91),
        "fit_range": (1.76, 1.98),
        "mean_limits": (1.82, 1.91),
        "sigma_limits": (0.005, 0.080),
        "rebin": [2, 2, 2, 3, 4, 5],
        "model": "gaus",
    },
    "antid0": {
        "label": "#bar{D}^{0}",
        "mass_title": "m_{K^{+}#pi^{-}} [GeV/c^{2}]",
        "cut": cut_antid0,
        "hist": "h_mass_AntiD0_vs_v0_pt",
        "draw_range": (1.70, 2.05),
        "sidebands": [(1.70, 1.78), (1.95, 2.05)],
        "seed_range": (1.82, 1.91),
        "fit_range": (1.76, 1.98),
        "mean_limits": (1.82, 1.91),
        "sigma_limits": (0.005, 0.080),
        "rebin": [2, 2, 2, 3, 4, 5],
        "model": "gaus",
    },
}

input_samples = [sample for sample in input_samples if sample.get("enabled", True)]

if not input_samples:
    raise RuntimeError("No enabled samples are configured in input_samples.")


In [3]:
# ============================================================
# Style and multi-file helpers
# ============================================================

ROOT_KEEPALIVE = []

def keep(obj):
    ROOT_KEEPALIVE.append(obj)
    root.SetOwnership(obj, False)
    return obj

def safe_root_name(text):
    return "".join(ch if ch.isalnum() else "_" for ch in str(text))

def Format_hist_loc(
    h, title, x_title, y_title,
    x_title_size=0.055, y_title_size=0.055,
    x_title_offset=0.95, y_title_offset=1.05,
    label_size_x=0.045, label_size_y=0.045,
    center_title=True
):
    h.SetTitle(title)
    h.GetXaxis().SetTitle(x_title)
    h.GetYaxis().SetTitle(y_title)
    h.GetXaxis().SetTitleSize(x_title_size)
    h.GetYaxis().SetTitleSize(y_title_size)
    h.GetXaxis().SetTitleOffset(x_title_offset)
    h.GetYaxis().SetTitleOffset(y_title_offset)
    h.GetXaxis().SetLabelSize(label_size_x)
    h.GetYaxis().SetLabelSize(label_size_y)
    if center_title:
        h.GetXaxis().CenterTitle()
        h.GetYaxis().CenterTitle()

input_files = {}

for sample in input_samples:
    full_name = file_path + sample["file_name"]
    input_file = root.TFile.Open(full_name)

    if not input_file or input_file.IsZombie():
        raise RuntimeError("Cannot open " + full_name)

    input_files[sample["name"]] = input_file
    print(f"Opened: {sample['label']} -> {full_name}")

def get_hist(sample_name, path):
    input_file = input_files[sample_name]
    h = input_file.Get(path)

    if not h:
        raise RuntimeError(
            f"Missing histogram in sample '{sample_name}': {path}"
        )

    clone_name = (
        safe_root_name(sample_name) + "_" +
        safe_root_name(path) + "_clone_" +
        str(len(ROOT_KEEPALIVE))
    )
    h = h.Clone(clone_name)
    h.SetDirectory(0)
    return keep(h)

def hist_path(cut_name, charge_name, hist_name):
    return f"{cut_name}/{charge_name}/{hist_name}"

def save_canvas(canvas, name):
    if safe_to_pdf:
        canvas.SaveAs(f"{outdir}/{name}.pdf")


Opened: 1st-order cor. -> input/v0s_pp_79513_newv0_seg_PR_full_pt02_r2_z15_q15_nc30_noCS0_v0.root
Opened: 2nd-order cor. -> input/v0s_pp_79523_newv0_PR_full_pt02_r2_z15_q15_nc30_noCS0_v1.root


In [4]:
# ============================================================
# Like-sign background and pT projections
# ============================================================

def project_mass(h2, ptmin, ptmax, name):
    b1 = h2.GetXaxis().FindBin(ptmin + 1.e-8)
    b2 = h2.GetXaxis().FindBin(ptmax - 1.e-8)
    h = h2.ProjectionY(name, b1, b2, "e")
    h.SetDirectory(0)
    return keep(h)

def sideband_integral(h, sidebands):
    value = 0.
    error2 = 0.
    for xmin, xmax in sidebands:
        b1 = h.GetXaxis().FindBin(xmin + 1.e-8)
        b2 = h.GetXaxis().FindBin(xmax - 1.e-8)
        for ibin in range(b1, b2 + 1):
            value += h.GetBinContent(ibin)
            error2 += h.GetBinError(ibin)**2
    return value, math.sqrt(error2)

def make_geometric_like_background(h_pp, h_mm, name):
    h = keep(h_pp.Clone(name))
    h.Reset("ICES")
    h.SetDirectory(0)

    for ibin in range(1, h.GetNbinsX() + 1):
        npp = h_pp.GetBinContent(ibin)
        nmm = h_mm.GetBinContent(ibin)
        epp = h_pp.GetBinError(ibin)
        emm = h_mm.GetBinError(ibin)

        if npp > 0. and nmm > 0.:
            h.SetBinContent(ibin, 2.*math.sqrt(npp*nmm))
            h.SetBinError(
                ibin,
                math.sqrt((nmm/npp)*epp*epp + (npp/nmm)*emm*emm)
            )
    return h

def scale_like_to_unlike(h_unlike, h_like, sidebands):
    fg, fg_err = sideband_integral(h_unlike, sidebands)
    bg, bg_err = sideband_integral(h_like, sidebands)

    scale = fg/bg if bg > 0. else 1.
    scale_err = (
        scale*math.sqrt((fg_err/fg)**2 + (bg_err/bg)**2)
        if fg > 0. and bg > 0. else 0.
    )

    h_scaled = keep(h_like.Clone(h_like.GetName() + "_scaled"))
    h_scaled.SetDirectory(0)
    h_scaled.Scale(scale)
    return h_scaled, scale, scale_err

def prepare_spectrum(sample, key, ptmin, ptmax, rebin, tag):
    cfg = species[key]
    sample_name = sample["name"]
    object_tag = safe_root_name(f"{sample_name}_{key}_{tag}")

    h2_unlike = get_hist(
        sample_name,
        hist_path(cfg["cut"], "unlike", cfg["hist"])
    )

    if like_background_method == "combined_like":
        h2_like = get_hist(
            sample_name,
            hist_path(cfg["cut"], "like", cfg["hist"])
        )
        h_like = project_mass(
            h2_like, ptmin, ptmax,
            f"h_{object_tag}_like"
        )
    else:
        h2_pp = get_hist(
            sample_name,
            hist_path(cfg["cut"], "plusplus", cfg["hist"])
        )
        h2_mm = get_hist(
            sample_name,
            hist_path(cfg["cut"], "minusminus", cfg["hist"])
        )
        h_pp = project_mass(
            h2_pp, ptmin, ptmax,
            f"h_{object_tag}_pp"
        )
        h_mm = project_mass(
            h2_mm, ptmin, ptmax,
            f"h_{object_tag}_mm"
        )
        h_like = make_geometric_like_background(
            h_pp, h_mm,
            f"h_{object_tag}_like"
        )

    h_fg = project_mass(
        h2_unlike, ptmin, ptmax,
        f"h_{object_tag}_fg"
    )

    if rebin > 1:
        h_fg.Rebin(rebin)
        h_like.Rebin(rebin)

    h_bg, scale, scale_err = scale_like_to_unlike(
        h_fg, h_like, cfg["sidebands"]
    )

    if normalize_for_shape_comparison:
        integral = h_fg.Integral()
        if integral > 0.:
            h_fg.Scale(1./integral)
            h_bg.Scale(1./integral)

    return {
        "sample": sample,
        "key": key,
        "ptmin": ptmin,
        "ptmax": ptmax,
        "fg": h_fg,
        "bg": h_bg,
        "scale": scale,
        "scale_err": scale_err,
    }


In [5]:
# ============================================================
# Signal + background fits
# ============================================================

def fit_gaussian_pol2(h, cfg, name):
    seed = keep(root.TF1(
        name + "_seed",
        "gaus",
        cfg["seed_range"][0],
        cfg["seed_range"][1]
    ))

    seed.SetParameters(
        max(h.GetBinContent(h.GetMaximumBin()), 1.),
        h.GetBinCenter(h.GetMaximumBin()),
        0.010
    )
    seed.SetParLimits(1, *cfg["mean_limits"])
    seed.SetParLimits(2, *cfg["sigma_limits"])
    h.Fit(seed, "RQ0")

    full = keep(root.TF1(
        name + "_full",
        "gaus(0)+pol2(3)",
        cfg["fit_range"][0],
        cfg["fit_range"][1]
    ))
    full.SetParameters(
        seed.GetParameter(0),
        seed.GetParameter(1),
        abs(seed.GetParameter(2)),
        0., 0., 0.
    )
    full.SetParLimits(1, *cfg["mean_limits"])
    full.SetParLimits(2, *cfg["sigma_limits"])
    h.Fit(full, "RQS0")

    signal = keep(root.TF1(
        name + "_signal",
        "gaus",
        cfg["fit_range"][0],
        cfg["fit_range"][1]
    ))
    signal.SetParameters(
        full.GetParameter(0),
        full.GetParameter(1),
        abs(full.GetParameter(2))
    )

    background = keep(root.TF1(
        name + "_background",
        "pol2",
        cfg["fit_range"][0],
        cfg["fit_range"][1]
    ))
    background.SetParameters(
        full.GetParameter(3),
        full.GetParameter(4),
        full.GetParameter(5)
    )

    return {
        "full": full,
        "signal": signal,
        "background": background,
        "mean": full.GetParameter(1),
        "sigma": abs(full.GetParameter(2)),
    }

def fit_voigt_pol2(h, cfg, name):
    # TMath::Voigt = Breit-Wigner convolved with a Gaussian.
    # The natural phi width is fixed; the Gaussian detector width is fitted.
    formula = "[0]*TMath::Voigt(x-[1],[2],[3],4)+pol2(4)"

    full = keep(root.TF1(
        name + "_full",
        formula,
        cfg["fit_range"][0],
        cfg["fit_range"][1]
    ))

    full.SetParameters(
        max(h.GetBinContent(h.GetMaximumBin()), 1.),
        1.019,
        0.003,
        cfg["gamma"],
        0., 0., 0.
    )
    full.SetParLimits(1, *cfg["mean_limits"])
    full.SetParLimits(2, *cfg["sigma_limits"])
    full.FixParameter(3, cfg["gamma"])
    h.Fit(full, "RQS0")

    signal = keep(root.TF1(
        name + "_signal",
        "[0]*TMath::Voigt(x-[1],[2],[3],4)",
        cfg["fit_range"][0],
        cfg["fit_range"][1]
    ))
    signal.SetParameters(
        full.GetParameter(0),
        full.GetParameter(1),
        abs(full.GetParameter(2)),
        cfg["gamma"]
    )

    background = keep(root.TF1(
        name + "_background",
        "pol2",
        cfg["fit_range"][0],
        cfg["fit_range"][1]
    ))
    background.SetParameters(
        full.GetParameter(4),
        full.GetParameter(5),
        full.GetParameter(6)
    )

    return {
        "full": full,
        "signal": signal,
        "background": background,
        "mean": full.GetParameter(1),
        "sigma": abs(full.GetParameter(2)),
    }

def fit_spectrum(result, name):
    cfg = species[result["key"]]
    return (
        fit_voigt_pol2(result["fg"], cfg, name)
        if cfg["model"] == "voigt"
        else fit_gaussian_pol2(result["fg"], cfg, name)
    )


In [6]:
# ============================================================
# Common multi-file overlay drawing function
# ============================================================

from sympy import latex

diffs = []
def draw_comparison(results, pad, name, integrated=False):
    if not results:
        return

    cfg = species[results[0]["key"]]

    pad.cd()
    root.gPad.SetLeftMargin(0.14)
    root.gPad.SetBottomMargin(0.14)
    root.gPad.SetRightMargin(0.02)
    root.gPad.SetTopMargin(0.02)

    y_title = "normalized pairs" if normalize_for_shape_comparison else "pairs"

    fits = []
    ymax = 1.

    for i, result in enumerate(results):
        sample = result["sample"]
        color = sample["color"]
        marker = sample["marker"]

        fit = fit_spectrum(
            result,
            safe_root_name(f"{name}_{sample['name']}")
        )
        result["fit"] = fit
        fits.append(fit)

        h_fg = result["fg"]
        h_bg = result["bg"]

        Format_hist_loc(h_fg, "", cfg["mass_title"], y_title)
        h_fg.GetXaxis().SetRangeUser(*cfg["draw_range"])

        h_fg.SetLineColor(color)
        h_fg.SetMarkerColor(color)
        h_fg.SetMarkerStyle(marker)
        h_fg.SetMarkerSize(0.75)
        h_fg.SetLineWidth(2)

        h_bg.SetLineColor(color)
        h_bg.SetMarkerColor(color)
        h_bg.SetMarkerStyle(marker)
        h_bg.SetMarkerSize(0.60)
        h_bg.SetLineStyle(3)
        h_bg.SetLineWidth(2)

        fit["full"].SetLineColor(color)
        fit["full"].SetLineWidth(3)

        fit["signal"].SetLineColor(color)
        fit["signal"].SetLineStyle(2)
        fit["signal"].SetLineWidth(2)

        fit["background"].SetLineColor(color)
        fit["background"].SetLineStyle(3)
        fit["background"].SetLineWidth(2)

        ymax = max(ymax, h_fg.GetMaximum(), h_bg.GetMaximum())

    frame = results[0]["fg"]
    frame.SetMinimum(0.)
    frame.SetMaximum(1.38*ymax)
    frame.Draw("E")

    if draw_scaled_like_sign:
        results[0]["bg"].Draw("E SAME")

    if draw_total_fit:
        results[0]["fit"]["full"].Draw("SAME")

    if draw_signal_component:
        results[0]["fit"]["signal"].Draw("SAME")

    for result in results[1:]:
        result["fg"].Draw("E SAME")

        if draw_scaled_like_sign:
            result["bg"].Draw("E SAME")

        if draw_total_fit:
            result["fit"]["full"].Draw("SAME")

        if draw_signal_component:
            result["fit"]["signal"].Draw("SAME")

    #drawing difference between first and second sample fgs:
    diff = results[1]["fg"].Rebin(1, results[0]["fg"].GetName() + "_diff")
    diff.Add(results[0]["fg"], -1)
    diff.SetLineColor(root.kGreen + 2)
    diff.SetMarkerColor(root.kGreen + 2)
    diff.SetMarkerStyle(20)
    diff.SetMarkerSize(0.75)
    diff.SetLineWidth(2)
    diffs.append(diff)
    if draw_diff:
        diffs[-1].Draw("E SAME")

    latex = keep(root.TLatex())
    latex.SetNDC()
    latex.SetTextFont(42)
    latex.SetTextSize(0.048)
    latex.DrawLatex(0.17, 0.92, f"{data_set}, {run_text}")
    latex.DrawLatex(0.17, 0.855, cfg["label"])

    if integrated:
        pt_text = (
            f"{results[0]['ptmin']:.1f} < p_{{T}} < "
            f"{results[0]['ptmax']:.1f} GeV/c, integrated"
        )
    else:
        pt_text = (
            f"{results[0]['ptmin']:.1f} < p_{{T}} < "
            f"{results[0]['ptmax']:.1f} GeV/c"
        )

    latex.DrawLatex(0.17, 0.79, pt_text)

    legend_y2 = 0.98
    legend_y1 = max(0.53, legend_y2 - 0.065*(len(results) + 1))
    legend = keep(root.TLegend(0.60, legend_y1, 0.94, legend_y2))
    legend.SetBorderSize(0)
    legend.SetFillStyle(0)
    legend.SetTextSize(0.045)

    for result in results:
        legend.AddEntry(
            result["fg"],
            result["sample"]["label"],
            "lep"
        )

    if draw_total_fit:
        legend.AddEntry(
            results[0]["fit"]["full"],
            "solid line: signal + pol2 fit",
            "l"
        )
    if draw_diff:
        legend.AddEntry(
            diffs[-1],
            "difference",
            "lep"
        )

    legend.Draw()

    # Show fitted mean and Gaussian resolution for each file.
    latex.SetTextSize(0.033)
    text_y = legend_y1 - 0.025

    for result in results:
        fit = result["fit"]
        latex.SetTextColor(result["sample"]["color"])

        #latex.DrawLatex(
        #    0.66, text_y,
        #    f"{result['sample']['label']}:"
        #)
        latex.DrawLatex(
            0.66, text_y,
            f"#mu = {fit['mean']:.5f} GeV/c^{{2}}"
        )
        latex.DrawLatex(
            0.66, text_y - 0.045,
            f"#sigma_{{G}} = {1000. * fit['sigma']:.2f} MeV/c^{{2}}"
        )

        text_y -= 0.145-0.045
    latex.SetTextColor(root.kBlack)
    root.gPad.RedrawAxis()


In [7]:
# ============================================================
# 1. All pT-integrated spectra: all files overlaid per pad
# ============================================================

integrated_results = {}

for key, cfg in species.items():
    integrated_results[key] = []

    for sample in input_samples:
        integrated_results[key].append(
            prepare_spectrum(
                sample,
                key,
                integrated_pt[0],
                integrated_pt[1],
                max(cfg["rebin"][0], 1),
                "integrated"
            )
        )

c_integrated = keep(root.TCanvas(
    "c_mass_integrated_comparison",
    "c_mass_integrated_comparison",
    1500,
    900
))
c_integrated.Divide(3, 2)

for i, key in enumerate(species):
    draw_comparison(
        integrated_results[key],
        c_integrated.cd(i + 1),
        f"fit_{key}_integrated_comparison",
        integrated=True
    )

c_integrated.Draw()
save_canvas(c_integrated, "mass_all_species_integrated_comparison")


In [8]:
# ============================================================
# 2. Spectra in pT bins: all files overlaid per pad
# ============================================================

differential_results = {}

for key, cfg in species.items():
    bins = pt_bins[key]
    differential_results[key] = []

    for i in range(len(bins) - 1):
        bin_results = []

        for sample in input_samples:
            bin_results.append(
                prepare_spectrum(
                    sample,
                    key,
                    bins[i],
                    bins[i + 1],
                    cfg["rebin"][i],
                    f"pt{i}"
                )
            )

        differential_results[key].append(bin_results)

    nrows = math.ceil(len(differential_results[key])/ncols)
    canvas = keep(root.TCanvas(
        f"c_mass_{key}_pt_comparison",
        f"c_mass_{key}_pt_comparison",
        1500,
        450*nrows
    ))
    canvas.Divide(ncols, nrows)

    for i, bin_results in enumerate(differential_results[key]):
        draw_comparison(
            bin_results,
            canvas.cd(i + 1),
            f"fit_{key}_pt{i}_comparison"
        )

    canvas.Draw()
    save_canvas(canvas, f"mass_{key}_pt_bins_comparison")

print("Finished multi-file comparison.")
print("Compared samples:")
for sample in input_samples:
    print(f"  - {sample['label']}: {sample['file_name']}")
print("PDF saving:", safe_to_pdf)
print("Output directory:", outdir)


Finished multi-file comparison.
Compared samples:
  - 1st-order cor.: v0s_pp_79513_newv0_seg_PR_full_pt02_r2_z15_q15_nc30_noCS0_v0.root
  - 2nd-order cor.: v0s_pp_79523_newv0_PR_full_pt02_r2_z15_q15_nc30_noCS0_v1.root
PDF saving: True
Output directory: output/v0_mass_selected_comparison


Warning in <Fit>: Fit data is empty 
Warning in <Fit>: Fit data is empty 
Warning in <TH1D::Rebin>: ngroup=3 is not an exact divider of nbins=350.
Warning in <TH1D::Rebin>: ngroup=3 is not an exact divider of nbins=350.
Warning in <TH1D::Rebin>: ngroup=3 is not an exact divider of nbins=350.
Warning in <TH1D::Rebin>: ngroup=3 is not an exact divider of nbins=350.
Warning in <TH1D::Rebin>: ngroup=4 is not an exact divider of nbins=350.
Warning in <TH1D::Rebin>: ngroup=4 is not an exact divider of nbins=350.
Warning in <TH1D::Rebin>: ngroup=4 is not an exact divider of nbins=350.
Warning in <TH1D::Rebin>: ngroup=4 is not an exact divider of nbins=350.
Warning in <TH1D::Rebin>: ngroup=3 is not an exact divider of nbins=350.
Warning in <TH1D::Rebin>: ngroup=3 is not an exact divider of nbins=350.
Warning in <TH1D::Rebin>: ngroup=3 is not an exact divider of nbins=350.
Warning in <TH1D::Rebin>: ngroup=3 is not an exact divider of nbins=350.
Warning in <TH1D::Rebin>: ngroup=4 is not an exact